# Promo Finder - live OpenAI run (Kuku Problem 1)
Runtime: **CPU** (no GPU). Run the cells in order. Cell 1 clones the repo from GitHub; cell 1b asks you to upload Kuku's scripts zip (confidential: it is not in the repo and only lives in this Colab session).

In [ ]:
#@title 1. Get the code from GitHub
REPO_URL = "https://github.com/Algoistricky2004/Promo-Finder.git"  #@param {type:"string"}
import shutil
shutil.rmtree("/content/kuku", ignore_errors=True)
!git clone -q "$REPO_URL" /content/kuku
%cd /content/kuku
!ls

In [ ]:
#@title 1b. Upload Kuku's scripts zip (the input pack)
import zipfile, os, shutil
from google.colab import files
up = files.upload()
shutil.rmtree("/content/_pack", ignore_errors=True)
for name in up:
    if name.endswith(".zip"): zipfile.ZipFile(name).extractall("/content/_pack")
os.makedirs("/content/kuku/pack", exist_ok=True)
for r, _, fs in os.walk("/content/_pack"):
    for f in fs:
        if f.endswith(".txt"): shutil.copy(os.path.join(r, f), "/content/kuku/pack/" + f)
print(len(os.listdir("/content/kuku/pack")), "script files in /content/kuku/pack (expect 50)")
%cd /content/kuku

In [ ]:
#@title 2. OpenAI API key (Colab Secrets named OPENAI_API_KEY, else you will be asked)
import os, getpass
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
print("key set:", bool(os.environ["OPENAI_API_KEY"]))

In [ ]:
#@title 3. Settings (model and mode are the only things to change to swap the model)
MODEL = "gpt-4o-mini"      #@param {type:"string"}
MODE = "select"            #@param ["select", "free"]
BASE_URL = "https://api.openai.com/v1"  #@param {type:"string"}
%cd /content/kuku

In [ ]:
#@title 4. Smoke test on episodes 1-3
!python run.py --pack ./pack --out outputs_live_test --judge openai --model "$MODEL" --mode "$MODE" --base-url "$BASE_URL" --episodes 1-3

In [ ]:
#@title 5. Full run, all 50 episodes
!python run.py --pack ./pack --out outputs_live --judge openai --model "$MODEL" --mode "$MODE" --base-url "$BASE_URL"

In [ ]:
#@title 6. Look at the result
import json
d = json.load(open("outputs_live/promos.json"))
print(len(d["promos"]), "promos kept,", len(d["rejected_by_validator"]), "dropped (duplicates / editor pass / invalid)\n")
for p in d["promos"][:10]:
    segs = " + ".join(f"Ep{s['ep']} {s['start']:.1f}-{s['end']:.1f}s" for s in p["segments"])
    print(f"#{p['rank']} ({p['runtime_s']}s) {segs}\n   hook: {p['hook']}\n   cliff: {p['cliffhanger']}\n   why: {p['why']}\n   editor: {p.get('editor_note','')}\n")
for r in d["rejected_by_validator"][:12]: print("DROPPED ep", r["ep"], r["errors"][0][:110])

In [ ]:
#@title 7. Quality check (code-only heuristic vs the judge)
!python analysis/heuristic_vs_judge.py ./pack outputs_live

In [ ]:
#@title 7b. (optional) Compare with the reference run: upload promos.json and no_promo_notes.json from the raw-outputs folder
from google.colab import files
up = files.upload()   # select 3_reference_run_claude_judged/promos.json AND 3_reference_run_claude_judged/saved_judge_stage/no_promo_notes.json
!python analysis/live_vs_reference.py outputs_live/promos.json promos.json no_promo_notes.json 36 > outputs_live/live_vs_reference.md
!cat outputs_live/live_vs_reference.md

In [ ]:
#@title 8. Download the live outputs
!cd /content/kuku && zip -qr raw_outputs_live.zip outputs_live
from google.colab import files; files.download("/content/kuku/raw_outputs_live.zip")